# Populate World-Camera Calibration Data

The goal of this notebook is to **generate the calibration assets under `data/` from the most primary available sources on Dropbox**. For camera measurements, that means raw sensor chunks and their metadata. For external measurements, that means the original instrument export or reference table. Checked-in TIFF and MAT outputs are never source material and must not be copied into place. They may be used only to validate a newly generated result.

Each calibration stage is organized into a **Function definitions** section followed by a **Configuration and execution** section. Running an execution cell always invokes its stage. With its `OVERWRITE_*` flag set to `False`, the stage preserves existing outputs and generates only missing outputs; setting that flag to `True` allows replacement. Paths, selections, and overwrite policy are kept together at each call site so the inputs to every stage are explicit.

Deprecated calibration artifacts, hidden operating-system files, generated implementation caches, and previously derived copies on Dropbox are intentionally outside the source boundary.

## Build outline and handoff

The intended execution order is:

1. **Declare primary sources.** Configure one Dropbox root and record the relative source path, source type, and selection rule for every output. Do not point at an existing generated TIFF or MAT file.
2. **Confirm the configured sources.** Each stage reads its configured source paths directly and lets the underlying file operation report missing or malformed inputs.
3. **Materialize `data/` inputs.** Extract raw frames directly when stable global indices are known. Reconstruct a temporary video only when the authoritative selection is expressed in video time or requires the normal frame-gap handling. Parse original instrument exports and reference tables into MAT files with explicit schemas.
4. **Run the calibration definitions.** Feed the newly generated `data/` inputs into the MATLAB and Python definition routines that write the corresponding files under `derived/`. Interactive decisions, such as checkerboard acceptance in Camera Calibrator, must be documented and preserved as reproducible selections where possible.
5. **Validate a clean rebuild.** Check file counts, array shapes, variable names, finite ranges, and important numerical summaries. Comparison with the checked-in assets is allowed here, but never during generation.

### Asset-by-asset plan

| Destination generated under `data/` | Most primary source | Generation |
| --- | --- | --- |
| `exampleWorldCameraImages/*_AGCandMS_*.mat` | FLIC_2001 indoor/outdoor and planetarium raw world/minispect chunks | Read the curated selection from the directory README, extract each selected raw Bayer frame directly by global index in the chunks (non-gap-filled), and write one MAT file per selection containing that frame, its AGC settings, and the nearest minispect sample on the shared clock. |
| `MacBethColorCheck/{indoor,outdoor}/<measurement>/lightLogger/close_AGCandMS_01.mat` | Numbered simultaneous Macbeth ColorChecker world/minispect recordings beneath the indoor and outdoor directories under the configured Dropbox root | Extract one physical raw-frame index from each numbered recording without counting gap-filled rows and write the same synchronized MAT schema used by `exampleWorldCameraImages`. |
| `AGCSettingsByNDF/worldCamera/NDF0/` through `NDF4/` | Five raw simultaneous world/minispect recordings under one Dropbox source directory | Read three physical raw-frame indices per NDF level from the directory README and write 15 synchronized MAT files using the same schema as `exampleWorldCameraImages`. |
| `fisheyeLensCalibration/intrinsics_calibration_images/` | The 47 accepted checkerboard TIFFs archived in Dropbox | Copy the saved calibration images directly because the original raw recording is incomplete. The reader must then use MATLAB's built-in Camera Calibrator app to fit the model, save the session, and export `derived/arducamB0392cameraIntrinsics.mat`; that interactive step is not automated here. |
| `flatFieldingFunction/rawFrames/` | Raw Fels Planetarium chunks | Reconstruct the recording with the documented gap and gain handling, extract the 36 documented time samples, then run `defineFlatFieldingFunction.m` to create `derived/flatFieldingFunction.mat` |
| `radiometricCorrectionRGB/rawFrames/` and cloudy-sky SPD | Raw simultaneous camera chunks plus the original PR670 export | Extract the ten uncorrected Bayer frames, parse the PR670 measurement, then run `defineRadiometricWeights.m` to create `derived/radiometricCorrectionRGB.mat` |
| `darkSignal/<AGC state>/*.tiff` | Covered-camera raw chunks for five fixed AGC states | Select ten documented raw frames per state, then run `defineDarkSignal.m` to create `derived/darkSignal.mat` |
| `camera_linearity_ND0_ND0p4_rgb_means.mat` | Original radiometric-calibration collection outputs | Run the existing conversion and camera-linearity analysis, then use the result in `defineFullWellCapacityEffect.m` |
| `empircalAGCAndIlluminance.mat` | Raw GKA world/minispect recordings | Derive `derived/MSIlluminanceToAGCLag.mat`, build the aligned point cloud, and compare it with the integrating-sphere lookup saved by `defineAGCToMeanLuminance.m` in `derived/cameraScoreToAverageLuminance.mat` |
| `agc_empirical_kernels.mat` | AGC simulation parameters and algorithm | Generate `derived/MSIlluminanceToAGCKernel.mat` with `defineMSIlluminanceToAGCKernel.py`; `defineMSIlluminanceToAGCLag.py` loads that standalone artifact |
| `ASM7341_spectralSensitivity.mat` and `IMX219_spectralSensitivity.mat` | Original manufacturer spreadsheet / published reference table | Parse the primary tables into stable MAT schemas used by downstream calibration code |


# Utility functions

Run this cell first. It imports the shared frame-context lookup and defines project paths plus the MAT and TIFF writers used by later stages.

In [2]:
from __future__ import annotations

import importlib
import re
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

import cv2
import numpy as np
from scipy.io import savemat


# Assume Jupyter was launched from code/defineWorldCameraCalibration/empiricalDataPrep
# to get the project root
NOTEBOOK_DIR: Path = Path.cwd().resolve()
PROJECT_ROOT: Path = NOTEBOOK_DIR.parents[2]

# Get the data directory, which we will populate 
DATA_ROOT: Path = PROJECT_ROOT / "data"

# Get the path to our utility libraries that we use 
CHUNK_IO_PATH: Path = (
    PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"
)
SENSOR_UTILITY_PATH: Path = PROJECT_ROOT / "code" / "library" / "sensor_utility"

# Add the libraries to our path and import them 
for library_path in (CHUNK_IO_PATH, SENSOR_UTILITY_PATH):
    assert library_path.exists(), f"Path: {library_path} does not exist"

    sys.path.append(str(library_path))
import chunk_io
import data_population_util
import world_util

# Create a utility function to write numbered frames as .tiff to an output path
def write_tiff_stack(
    frames: np.ndarray | list[np.ndarray], output_dir: Path, *, overwrite: bool
) -> None:
    """Write a sequential stack of uncompressed TIFF images.

    Parameters
    ----------
    frames
        Image arrays written in iteration order.
    output_dir
        Directory that receives ``0.tiff``, ``1.tiff``, and so on.
    overwrite
        Whether an existing numbered TIFF may be replaced.

    Returns
    -------
    None

    Raises
    ------
    IOError
        If OpenCV cannot write an output image.
    """
    
    # Ensure the output directory exists (parent directories included)
    output_dir.mkdir(parents=True, exist_ok=True)

    # Enumerate over the frames 
    for frame_index, frame in enumerate(frames):
        # Construct the output path
        output_path: Path = output_dir / f"{frame_index}.tiff"

        # If the frame already exists and we don't want to overwrite 
        # existing, just skip
        if output_path.exists() and not overwrite:
            continue

        # Write the frame 
        written: bool = cv2.imwrite(
            str(output_path), frame, [cv2.IMWRITE_TIFF_COMPRESSION, 1] # write the tiff with NO compression
        )

        # Catch errors with writing 
        if not written:
            raise IOError(f"OpenCV could not write {output_path}")


def write_frame_context_mat(
    frame_context: dict[str, Any],
    output_path: Path,
    *,
    readme: str,
    source_image_filename: str,
    global_frame_index: int,
) -> None:
    """Add provenance fields and save synchronized frame context as MAT.

    Parameters
    ----------
    frame_context
        World-camera and minispect values returned by ``find_target_frame``.
    output_path
        Destination MAT-file path.
    readme
        Human-readable description stored in the MAT payload.
    source_image_filename
        Descriptive label for the selected source frame.
    global_frame_index
        Zero-based physical world-frame index used for the selection.

    Returns
    -------
    None
    """
    # Create only the destination directory owned by the calling stage.
    output_path.parent.mkdir(parents=True, exist_ok=True)
    # Combine caller-owned provenance with the context returned by the reader.
    mat_data: dict[str, Any] = {
        "README": readme,
        "sourceImageFilename": source_image_filename,
        "globalWorldFrameIndex": np.int64(global_frame_index),
        **frame_context,
    }
    # Serialize the complete measurement using the established MAT options.
    savemat(
        output_path, mat_data, do_compression=True, long_field_names=True
    )

# `data/exampleWorldCameraImages/`

**What this is:** A small curated gallery used to inspect representative world-camera images from indoor, outdoor, and planetarium conditions. The current archive contains three indoor frames, three outdoor frames, and one planetarium frame, each stored as one `*_AGCandMS_*.mat` file holding the completely raw Bayer frame together with that frame's AGC settings and the nearest minispect sample. No processing is applied.

**Where it comes from:** Frames were manually selected from the FLIC_2001 `walkIndoor` and `walkOutdoor` recordings, the FLIC_1054 `sitBiopond` recording, and the planetarium recording; `outdoor3` is explicitly raw frame 5000. The visual choice is not algorithmic, so the selection itself lives in the directory README as a table of zero-based global raw-frame indices. That table is the durable, reproducible selection rule.

**Output:** `data/exampleWorldCameraImages/*_AGCandMS_*.mat`.

## Rebuild the synchronized metadata from the README

Run the configuration and execution cell to regenerate the example MAT files. It reads the selected raw-frame indices out of the directory README and writes one MAT file per selection containing the raw frame, its AGC settings, and the nearest minispect sample.

## Function definitions

In [ ]:
# -----------------------------------------------------------------------------
# README parser | NOTE: AI Generated
# Read the marker-delimited table with regex and validate it before any raw data
# are opened, so missing, duplicate, or unexpected frame rows fail immediately.
# -----------------------------------------------------------------------------
def read_example_frame_indices_from_readme(
    readme_path: Path, expected_names: set[str]
) -> dict[str, int]:
    """Read and validate curated example-frame indices from a README.

    Parameters
    ----------
    readme_path
        README containing the marker-delimited frame-selection table.
    expected_names
        Exact set of example labels that the table must contain.

    Returns
    -------
    dict[str, int]
        Mapping from labels such as ``indoor1`` to global frame indices.

    Raises
    ------
    ValueError
        If the table is missing, duplicated, or differs from the expected labels.
    """
    # Define the markers that isolate the frame-selection table.
    block_pattern: re.Pattern[str] = re.compile(
        r"<!-- populateData:example-frame-indices:start -->(?P<table>.*?)"
        r"<!-- populateData:example-frame-indices:end -->",
        re.DOTALL,
    )
    # Define the row format that carries each example label and raw-frame index.
    row_pattern: re.Pattern[str] = re.compile(
        r"^\|\s*`(?P<name>[A-Za-z]+\d+)`\s*\|[^|]*\|\s*(?P<index>\d+)\s*\|$",
        re.MULTILINE,
    )
    # Read the durable selection rule from the configured README.
    readme_text: str = readme_path.read_text(encoding="utf-8")
    # Require exactly one marker-delimited selection table.
    blocks: list[re.Match[str]] = list(block_pattern.finditer(readme_text))
    if len(blocks) != 1:
        raise ValueError("The README must contain exactly one generated example-frame-index block.")
    matches: list[re.Match[str]] = list(
        row_pattern.finditer(blocks[0].group("table"))
    )
    frame_indices: dict[str, int] = {
        match.group("name"): int(match.group("index")) for match in matches
    }
    if len(frame_indices) != len(matches):
        raise ValueError("The README contains duplicate example-frame rows.")
    # Confirm that the README and source-path map describe the same selections.
    if set(frame_indices) != expected_names:
        missing: list[str] = sorted(expected_names - set(frame_indices))
        extra: list[str] = sorted(set(frame_indices) - expected_names)
        raise ValueError(f"README example-frame rows do not match the curated set; missing={missing}, extra={extra}")
    return frame_indices


# -----------------------------------------------------------------------------
# Output generation
# Parse the README, decide which MAT files are allowed to be written, retrieve
# the needed contexts, then save missing files without touching protected outputs.
# -----------------------------------------------------------------------------
def populate_example_world_camera_measurements(
    raw_data_sources: dict[str, Path],
    output_dir: Path,
    mat_readme: str,
    *,
    overwrite: bool = False,
) -> None:
    """Generate synchronized MAT files for the curated example frames.

    Parameters
    ----------
    raw_data_sources
        Mapping from each README label to its raw recording directory.
    output_dir
        Directory containing the selection README and receiving MAT files.
    mat_readme
        Description stored in each generated MAT payload.
    overwrite
        Whether existing MAT files may be replaced.

    Returns
    -------
    None
    """
    
    
    # Read in the frame indices for each of the example world images
    # from the README
    # Output format will be indoor1 : IDX, indoor2: IDX 
    frame_indices: dict[str, int] = read_example_frame_indices_from_readme(
        output_dir / "README.md", set(raw_data_sources)
    )
    
    
    # Construct each MAT output filename directly from labels, such as indoor1.
    output_paths: dict[str, Path] = {}
    for example_image_name in frame_indices:
        # Separate the condition text from the trailing selection number.
        label_match: re.Match[str] | None = re.fullmatch(
            r"(?P<condition>[A-Za-z]+)(?P<sequence>\d+)", example_image_name
        )
        if label_match is None:
            raise ValueError(f"Unsupported example-frame label: {example_image_name}")
        # Preserve the established condition_AGCandMS_XX.mat output convention.
        output_paths[example_image_name] = output_dir / (
            f"{label_match.group('condition')}_AGCandMS_"
            f"{int(label_match.group('sequence')):02d}.mat"
        )
    
    # Create the destination before writing MAT files
    output_dir.mkdir(parents=True, exist_ok=True)

    # Output the desired images using our helper function 
    # to find world frames + MS info by global frame index 
    for example_image_name, global_frame_index in frame_indices.items():
        # Skip existing output unless we want to overwrite them
        if not overwrite and output_paths[example_image_name].exists():
            continue

        # Retrieve the selected raw frame and synchronized sensor context.
        frame_context: dict[str, Any] = data_population_util.find_target_frame(
            raw_data_sources[example_image_name],
            global_frame_index,
        )
        # Add example-specific provenance and save the completed MAT payload.
        write_frame_context_mat(
            frame_context,
            output_paths[example_image_name],
            source_image_filename=example_image_name,
            readme=mat_readme,
            global_frame_index=global_frame_index,
        )

    
    return 

## Configuration and execution

In [ ]:
# Choose whether existing synchronized MAT files may be replaced.
OVERWRITE_EXAMPLE_MEASUREMENTS: bool = False

# Place all generated example MAT files beside their selection README.
EXAMPLE_OUTPUT_DIR: Path = DATA_ROOT / "exampleWorldCameraImages"

# Map each README frame label to the raw recording that contains it.
EXAMPLE_RAW_DATA_SOURCES: dict[str, Path] = {
    "indoor1": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA/1"),
    "indoor2": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA/1"),
    "indoor3": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA/1"),
    "outdoor1": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA/1"),
    "outdoor2": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA/1"),
    "outdoor3": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_1054/sitBiopond/GKA/1"),
    "planetarium1": Path("/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/flatFieldingFunction/planetarium_fielding_function_raw"),
}

# Describe the synchronized variables stored inside every generated MAT file.
EXAMPLE_MAT_README: str = (
    "Synchronized context for one curated raw world-camera example. "
    "sourceImageFilename is the frame's label in the directory README; globalWorldFrameIndex is its zero-based "
    "index across naturally ordered raw world chunks; worldTimestampSeconds is on the "
    "shared logger clock; worldFrame is the unprocessed Bayer frame; AGCSettings always contains "
    "the three settings Again, Dgain, and exposure that the reconstruction pipeline consumes, "
    "normalized from whichever schema the source recording used; minispectTimestampSeconds and "
    "minispectValue contain the nearest minispect packet and its parsed AS, TS, LS, and TEMP values."
)

# Run the stage with the configured paths, metadata description, and overwrite policy.
populate_example_world_camera_measurements(
    EXAMPLE_RAW_DATA_SOURCES,
    EXAMPLE_OUTPUT_DIR,
    EXAMPLE_MAT_README,
    overwrite=OVERWRITE_EXAMPLE_MEASUREMENTS,
)

# `data/MacBethColorCheck/{indoor,outdoor}/<measurement>/lightLogger/`

**What this is:** Raw world-camera views of the Macbeth ColorChecker from one or more numbered indoor and outdoor measurements. Every MAT file has the same synchronized schema as the files in `data/exampleWorldCameraImages/`: the unprocessed Bayer frame, normalized AGC settings, exact world timestamp, and nearest minispect sample.

**Where it comes from:** The configured Dropbox `MacBethColorCheck` directory contains `indoor/` and `outdoor/`; each condition contains numbered measurement directories (`1/`, `2/`, and so on), and each numbered directory contains `MacBethColorCheck_raw/`.

**Selection rule:** Read the condition, measurement number, and physical frame index directly from the Selected Light Logger Frames table in `data/MacBethColorCheck/README.md`. Indices count only frames physically present in naturally ordered raw chunks; gap-filled metadata rows are excluded before indexing.

**Output:** One `data/MacBethColorCheck/<condition>/<measurement>/lightLogger/close_AGCandMS_01.mat` per README row. Each numbered measurement directory also has a `PR670/` directory for its separately archived reference spectra.

## Function definitions

In [ ]:
"""
NOTE: AI Generated
"""
def read_macbeth_frame_selections(
    readme_path: Path, expected_conditions: tuple[str, ...]
) -> dict[tuple[str, int], int]:
    """Read one physical frame index per Macbeth measurement.

    Parameters
    ----------
    readme_path
        Macbeth README containing the selected Light Logger frame table.
    expected_conditions
        Lighting conditions that must each appear in the table.

    Returns
    -------
    dict[tuple[str, int], int]
        Mapping from ``(condition, measurement)`` to global frame index.

    Raises
    ------
    ValueError
        If the section is missing, rows are duplicated, or conditions differ.
    """
    # Read the durable condition, measurement, and frame-index table.
    text = readme_path.read_text(encoding="utf-8")
    # Isolate only the selected Light Logger frame section.
    section = re.search(
        r"^## Selected Light Logger Frames\s*\n(.*?)(?=^## |\Z)",
        text, flags=re.MULTILINE | re.DOTALL,
    )
    if section is None:
        raise ValueError(f"Selected Light Logger Frames section missing in {readme_path}")
    selections: dict[tuple[str, int], int] = {}
    # Parse each indoor/outdoor measurement row from the Markdown table.
    rows = re.findall(
        r"^\|\s*(indoor|outdoor)\s*\|\s*(\d+)\s*\|\s*(\d+)\s*\|",
        section.group(1), flags=re.MULTILINE | re.IGNORECASE,
    )
    for condition_text, measurement_text, frame_index_text in rows:
        key = (condition_text.lower(), int(measurement_text))
        if key in selections:
            raise ValueError(f"Duplicate Macbeth selection for {key} in {readme_path}")
        selections[key] = int(frame_index_text)
    # Require at least one selection for every configured lighting condition.
    if not selections or set(condition for condition, _ in selections) != set(expected_conditions):
        raise ValueError(f"Expected at least one selection per condition in {readme_path}")
    return selections


def populate_macbeth_world_camera_measurements(
    raw_root: Path,
    output_dir: Path,
    conditions: tuple[str, ...],
    mat_readme: str,
    *,
    overwrite: bool = False,
) -> None:
    """Generate synchronized MAT files for numbered Macbeth recordings.

    Parameters
    ----------
    raw_root
        Root containing condition and numbered measurement directories.
    output_dir
        Macbeth data directory containing the selection README.
    conditions
        Lighting conditions required by the selection table.
    mat_readme
        Description stored in each generated MAT payload.
    overwrite
        Whether existing measurement MAT files may be replaced.

    Returns
    -------
    None
    """
    # Read the fixed frame selected for each numbered recording.
    # Form: (condition, measurement), frame
    frame_indices: dict[tuple[str, int], int] = read_macbeth_frame_selections(
        output_dir / "README.md", conditions
    )

    # Process each condition and measurement in README table order.
    for (condition, measurement), global_frame_index in frame_indices.items():
        # Construct this numbered measurement's MAT destination directly.
        output_path: Path = (
            output_dir / condition / str(measurement)
            / "lightLogger" / "close_AGCandMS_01.mat"
        )
        # Skip existing measurements unless we want to overwrite
        if output_path.exists() and not overwrite:
            continue

        # Create the numbered measurement's lightLogger directory.
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Construct the corresponding Dropbox raw-chunk directory directly.
        raw_chunks: Path = (
            raw_root / condition / str(measurement) / "MacBethColorCheck_raw"
        )

        # Retrieve the selected raw frame and synchronized sensor context.
        frame_context: dict[str, Any] = data_population_util.find_target_frame(
            raw_chunks,
            global_frame_index,
        )
        # Add measurement-specific provenance and save the completed MAT payload.
        write_frame_context_mat(
            frame_context,
            output_path,
            source_image_filename=f"{condition}{measurement}_close_1.tiff",
            readme=mat_readme,
            global_frame_index=global_frame_index,
        )


    return

## Configuration and execution

In [3]:
# Choose whether existing synchronized Macbeth MAT files may be replaced.
OVERWRITE_MACBETH_MEASUREMENTS: bool = False
# Point to the common Dropbox root and the repository output directory.
MACBETH_RAW_ROOT: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "MacBethColorCheck"
)
MACBETH_OUTPUT_DIR: Path = DATA_ROOT / "MacBethColorCheck"
# Require selections for both supported lighting conditions.
MACBETH_CONDITIONS: tuple[str, ...] = ("indoor", "outdoor")
# Describe the synchronized variables stored inside every generated MAT file.
MACBETH_MAT_README: str = (
    "Synchronized context for one selected close-view raw Macbeth ColorChecker frame. "
    "sourceImageFilename is the historical frame label; globalWorldFrameIndex "
    "is its zero-based index across naturally ordered raw world chunks without "
    "gap-filled frames; worldTimestampSeconds is on the shared logger clock; "
    "worldFrame is the unprocessed Bayer frame; AGCSettings contains Again, "
    "Dgain, and exposure; minispectTimestampSeconds and minispectValue contain "
    "the nearest minispect packet and its parsed AS, TS, LS, and TEMP values."
)

# Run the stage with all paths, conditions, documentation, and policy visible.
populate_macbeth_world_camera_measurements(
    MACBETH_RAW_ROOT,
    MACBETH_OUTPUT_DIR,
    MACBETH_CONDITIONS,
    MACBETH_MAT_README,
    overwrite=OVERWRITE_MACBETH_MEASUREMENTS,
)

Generated 1 Macbeth MAT files; preserved 2 existing outputs in /Users/zacharykelly/Documents/MATLAB/projects/lightLoggerAnalysis/data/MacBethColorCheck


# `data/AGCSettingsByNDF/`

**What this is:** Three synchronized raw world-camera measurements for each NDF level from 0 through 4. Every MAT file uses the same schema as `data/exampleWorldCameraImages`: the raw Bayer frame, normalized AGC settings, exact world timestamp, and nearest minispect sample.

**Where it comes from:** One Dropbox source directory containing five raw world/minispect recordings named `0NDF` through `4NDF`. The configured source below points to the local synced Dropbox directory.

**Selection rule:** `data/AGCSettingsByNDF/README.md` records three zero-based physical frame indices for each NDF level. The current indices are placeholder zeros. Gap-filled metadata rows are excluded before indexing.

**Output:** Three `*_AGCandMS_*.mat` files in each of `data/AGCSettingsByNDF/worldCamera/NDF0/` through `NDF4/`. The `PR670/` directory is scaffolded for the corresponding spectroradiometer data but is not populated by this stage.

## Function definitions

In [ ]:
"""
    NOTE: AI Generated
"""
def read_agc_ndf_frame_indices_from_readme(
    readme_path: Path, ndf_levels: tuple[int, ...], selections_per_level: int
) -> dict[tuple[int, int], int]:
    """Read and validate physical frame indices for every NDF level.

    Parameters
    ----------
    readme_path
        README containing the marker-delimited AGC-by-NDF table.
    ndf_levels
        Exact NDF levels expected in the table.
    selections_per_level
        Number of frame selections expected for each NDF level.

    Returns
    -------
    dict[tuple[int, int], int]
        Mapping from ``(NDF level, selection number)`` to frame index.

    Raises
    ------
    ValueError
        If the table is missing, duplicated, or has unexpected selections.
    """
    # Define the marker-delimited table and compact NDF-row formats.
    block_pattern: re.Pattern[str] = re.compile(
        r"<!-- populateData:agc-settings-by-ndf-frame-indices:start -->(?P<table>.*?)"
        r"<!-- populateData:agc-settings-by-ndf-frame-indices:end -->",
        re.DOTALL,
    )
    row_pattern: re.Pattern[str] = re.compile(
        r"^\|\s*(?P<ndf>\d+)\s*\|\s*(?P<frame1>\d+)\s*\|"
        r"\s*(?P<frame2>\d+)\s*\|\s*(?P<frame3>\d+)\s*\|$",
        re.MULTILINE,
    )
    # Read the fixed selection table from the configured README.
    readme_text: str = readme_path.read_text(encoding="utf-8")
    blocks: list[re.Match[str]] = list(
        block_pattern.finditer(readme_text)
    )
    if len(blocks) != 1:
        raise ValueError(
            "The AGC-by-NDF README must contain exactly one frame-index block."
        )

    matches: list[re.Match[str]] = list(
        row_pattern.finditer(blocks[0].group("table"))
    )
    parsed_levels: list[int] = [int(match.group("ndf")) for match in matches]
    if len(set(parsed_levels)) != len(matches):
        raise ValueError("The AGC-by-NDF README contains duplicate NDF rows.")
    if set(parsed_levels) != set(ndf_levels):
        missing_levels: list[int] = sorted(set(ndf_levels) - set(parsed_levels))
        extra_levels: list[int] = sorted(set(parsed_levels) - set(ndf_levels))
        raise ValueError(
            "AGC-by-NDF README rows do not match the expected NDF levels; "
            f"missing={missing_levels}, extra={extra_levels}"
        )

    # Expand each compact README row into the (NDF level, selection number)
    # keys used by the generation loop below.
    frame_indices: dict[tuple[int, int], int] = {
        (int(match.group("ndf")), selection_number): int(
            match.group(f"frame{selection_number}")
        )
        for match in matches
        for selection_number in range(1, selections_per_level + 1)
    }

    expected_selections: set[tuple[int, int]] = {
        (ndf_level, selection_number)
        for ndf_level in ndf_levels
        for selection_number in range(1, selections_per_level + 1)
    }
    if set(frame_indices) != expected_selections:
        missing: list[tuple[int, int]] = sorted(
            expected_selections - set(frame_indices)
        )
        extra: list[tuple[int, int]] = sorted(
            set(frame_indices) - expected_selections
        )
        raise ValueError(
            "AGC-by-NDF README rows do not match the expected 15 selections; "
            f"missing={missing}, extra={extra}"
        )
    return frame_indices


def populate_agc_ndf_world_camera_measurements(
    raw_chunks_by_level: dict[int, Path],
    output_root: Path,
    ndf_levels: tuple[int, ...],
    selections_per_level: int,
    mat_readme: str,
    *,
    overwrite: bool = False,
) -> None:
    """Generate synchronized world-camera MAT files for each NDF level.

    Parameters
    ----------
    raw_chunks_by_level
        Mapping from NDF level to its raw recording directory.
    output_root
        AGC-by-NDF data directory containing the selection README.
    ndf_levels
        NDF levels that must be represented.
    selections_per_level
        Number of selected frames generated for each NDF level.
    mat_readme
        Description stored in each generated MAT payload.
    overwrite
        Whether existing selection MAT files may be replaced.

    Returns
    -------
    None
    """
    
    # First, let's get the frame indices for each NDF level and each 
    # measurement at that level 
    # Format: (NDF, frame): index
    frame_indices: dict[tuple[int, int], int] = (
        read_agc_ndf_frame_indices_from_readme(
            output_root / "README.md", ndf_levels, selections_per_level
        )
    )
    # Place generated camera measurements beneath the worldCamera directory.
    world_output_root: Path = output_root / "worldCamera"
    
    
    # Process every NDF selection in stable level/selection order.
    for (ndf_level, selection_number), global_frame_index in sorted(
        frame_indices.items()
    ):
        # Construct this selection's output path
        output_path: Path = (
            world_output_root
            / f"NDF{ndf_level}"
            / f"NDF{ndf_level}_AGCandMS_{selection_number:02d}.mat"
        )

        # Skip existing measurements unless we want to overwrite
        if output_path.exists() and not overwrite:
            continue
        
        source_image_filename: str = (
            f"NDF{ndf_level}_{selection_number}.tiff"
        )
        # Retrieve the selected raw frame and synchronized sensor context.
        frame_context: dict[str, Any] = data_population_util.find_target_frame(
            raw_chunks_by_level[ndf_level],
            global_frame_index,
        )
        
        # Add NDF-specific provenance and save the completed MAT payload.
        write_frame_context_mat(
            frame_context,
            output_path,
            source_image_filename=source_image_filename,
            readme=mat_readme,
            global_frame_index=global_frame_index,
        )

    return

## Configuration and execution

In [ ]:
# Choose whether existing synchronized AGC-by-NDF MAT files may be replaced.
OVERWRITE_AGC_NDF_MEASUREMENTS: bool = False
# Point to the common Dropbox source and repository output roots.
AGC_NDF_RAW_ROOT: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "AGCSettingsByNDF"
)
AGC_NDF_OUTPUT_ROOT: Path = DATA_ROOT / "AGCSettingsByNDF"

# Configure five NDF levels with three selected frames per level.
AGC_NDF_LEVELS: tuple[int, ...] = tuple(range(5))
AGC_NDF_SELECTIONS_PER_LEVEL: int = 3

# Map each NDF level to its raw world/minispect recording.
AGC_NDF_RAW_CHUNKS_BY_LEVEL: dict[int, Path] = {
    ndf_level: AGC_NDF_RAW_ROOT / f"{ndf_level}NDF"
    for ndf_level in AGC_NDF_LEVELS
}
# Describe the synchronized variables stored inside every generated MAT file.
AGC_NDF_MAT_README: str = (
    "Synchronized context for one selected raw world-camera frame in the "
    "AGC-by-NDF measurement. sourceImageFilename identifies its NDF level "
    "and selection number; globalWorldFrameIndex is its zero-based index "
    "across captured raw world frames after gap-filled metadata rows are "
    "excluded; worldTimestampSeconds is on the shared logger clock; "
    "worldFrame is the unprocessed Bayer frame; AGCSettings contains Again, "
    "Dgain, and exposure; minispectTimestampSeconds and minispectValue contain "
    "the nearest minispect packet and its parsed AS, TS, LS, and TEMP values."
)

# Run the stage with all paths, dimensions, documentation, and policy visible.
populate_agc_ndf_world_camera_measurements(
    AGC_NDF_RAW_CHUNKS_BY_LEVEL,
    AGC_NDF_OUTPUT_ROOT,
    AGC_NDF_LEVELS,
    AGC_NDF_SELECTIONS_PER_LEVEL,
    AGC_NDF_MAT_README,
    overwrite=OVERWRITE_AGC_NDF_MEASUREMENTS,
)

# `data/fisheyeLensCalibration/`

**What this is:** The checkerboard images used as input to MATLAB's built-in Camera Calibrator app to estimate focal length, principal point, and fisheye distortion for the ArduCam B0392 IMX219 world camera.

**Where it comes from:** The 47 accepted checkerboard TIFFs archived directly in Dropbox under `FOV_and_intrinsics/intrinsics_calibration/intrinsics_calibration_images/`. The original raw recording cannot be reconstructed completely because metadata uploaded for several chunks whose corresponding frame arrays are missing. Rather than manually attempting to rediscover indices from an incomplete recording, this stage copies the saved accepted frames directly.

**What it does:** Copies every archived TIFF into `data/fisheyeLensCalibration/intrinsics_calibration_images/`, preserving existing outputs unless overwrite is enabled. This notebook stops after producing the input TIFFs.

**Required manual MATLAB step:** After the TIFFs have been generated, open MATLAB's built-in Camera Calibrator app (for example, by running `cameraCalibrator`), load `data/fisheyeLensCalibration/intrinsics_calibration_images/`, review checkerboard acceptance, fit the camera model, and export the resulting intrinsics to `derived/arducamB0392cameraIntrinsics.mat`. No MATLAB code or GUI automation is run by this notebook.

**Notebook output:** `data/fisheyeLensCalibration/intrinsics_calibration_images/`.

## Function definitions

In [ ]:
# -----------------------------------------------------------------------------
# Archived calibration-image copy
# Preserve the accepted filenames and file metadata while copying directly from
# the durable Dropbox archive. Numeric TIFF names are copied in numeric order.
# -----------------------------------------------------------------------------
def populate_fisheye_calibration_images(
    source_images: Path, image_output: Path, *, overwrite: bool = False
) -> None:
    """Copy archived accepted checkerboard TIFFs into the data directory.

    Parameters
    ----------
    source_images
        Directory containing the accepted checkerboard TIFF files.
    image_output
        Directory that receives the copied calibration images.
    overwrite
        Whether an existing destination image may be replaced.

    Returns
    -------
    None
    """
    # Collect only TIFF files from the source directory.
    source_paths: list[Path] = []
    for path in source_images.iterdir():
        if path.is_file() and path.suffix.lower() in {".tif", ".tiff"}:
            source_paths.append(path)

    # Separate numeric names from any descriptive filenames.
    numeric_paths: list[Path] = [
        path for path in source_paths if path.stem.isdigit()
    ]
    descriptive_paths: list[Path] = [
        path for path in source_paths if not path.stem.isdigit()
    ]
    # Sort numeric images by frame number, then append named images alphabetically.
    numeric_paths.sort(key=lambda path: int(path.stem))
    descriptive_paths.sort(key=lambda path: path.name.lower())
    source_paths = numeric_paths + descriptive_paths
    
    # Create the destination directory before copying accepted images.
    image_output.mkdir(parents=True, exist_ok=True)

    # Copy each archived image while retaining its filename and file metadata.
    for source_path in source_paths:
        output_path: Path = image_output / source_path.name

        # Skip existing images unless replacement is enabled.
        if output_path.exists() and not overwrite:
            continue
        
        # copy2 preserves the source file's timestamps and other metadata.
        shutil.copy2(source_path, output_path)
        
    return 

## Configuration and execution

In [ ]:
# Choose whether existing accepted calibration images may be replaced.
OVERWRITE_FISHEYE_IMAGES: bool = False

# Point to the durable Dropbox archive of accepted checkerboard TIFFs.
FISHEYE_SOURCE_IMAGES: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "FOV_and_intrinsics/intrinsics_calibration/intrinsics_calibration_images"
)
# Store the copied TIFFs in the calibration data directory.
FISHEYE_IMAGE_OUTPUT: Path = (
    DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_images"
)

# Run the copy stage with the configured source, destination, and overwrite policy.
populate_fisheye_calibration_images(
    FISHEYE_SOURCE_IMAGES,
    FISHEYE_IMAGE_OUTPUT,
    overwrite=OVERWRITE_FISHEYE_IMAGES,
)

# `data/flatFieldingFunction/`

**What this is:** Raw Bayer frames used to estimate the spatial sensitivity imposed by the fisheye lens. The camera pointed at the nominally uniform Fels Planetarium dome and was rotated about its optical axis. Averaging frames across orientations reduces dome-specific spatial structure while preserving camera/lens structure.

**Where it comes from:** World-camera chunks stored in the lab Dropbox under `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/fielding_function/planetarium_fielding_function_raw`. These are the paths and selections recovered from `code/library/matlabIO/python_libraries/scratch2.ipynb`.

**What it does:** Converts the chunks to a temporary grayscale video with digital gain applied, verifies the documented 180 fps frame rate, and extracts the 36 time samples currently consumed by `defineFlatFieldingFunction.m`. Files are named sequentially in selection order, not by original video-frame number.

**Output:** `data/flatFieldingFunction/rawFrames/0.tiff` through `35.tiff`. `defineFlatFieldingFunction.m` linearizes and averages these frames, fits the flattened Gaussian, and writes `derived/flatFieldingFunction.mat`.

## Function definitions

In [ ]:
def read_flat_field_selection_from_readme(
    readme_path: Path,
) -> tuple[float, list[int], list[int]]:
    """Read and validate the flat-field video-frame selection.

    Parameters
    ----------
    readme_path
        README containing the frame rate and marker-delimited selection table.

    Returns
    -------
    tuple[float, list[int], list[int]]
        Recording FPS, selected times in seconds, and video-frame indices.

    Raises
    ------
    ValueError
        If the block, frame rate, rows, ordering, or index calculation is invalid.
    """
    # Read the README once before isolating the generated selection block.
    readme_text: str = readme_path.read_text(encoding="utf-8")
    block_pattern: re.Pattern[str] = re.compile(
        r"<!-- populateData:flat-field-frame-selection:start -->(?P<table>.*?)"
        r"<!-- populateData:flat-field-frame-selection:end -->",
        re.DOTALL,
    )
    blocks: list[re.Match[str]] = list(block_pattern.finditer(readme_text))
    if len(blocks) != 1:
        raise ValueError("The flat-field README must contain exactly one selection block.")
    selection_text: str = blocks[0].group("table")

    # Read the reconstruction frame rate recorded above the selection table.
    fps_match: re.Match[str] | None = re.search(
        r"Recording frame rate:\s*`(?P<fps>\d+(?:\.\d+)?)`\s*fps",
        selection_text,
    )
    if fps_match is None:
        raise ValueError("The flat-field selection block has no frame rate.")
    expected_fps: float = float(fps_match.group("fps"))

    # Parse output number, selected time, and temporary-video index per row.
    rows: list[tuple[str, str, str]] = re.findall(
        r"^\|\s*`(?P<output>\d+)\.tiff`\s*\|\s*(?P<time>\d+)\s*"
        r"\|\s*(?P<index>\d+)\s*\|$",
        selection_text,
        flags=re.MULTILINE,
    )
    if not rows:
        raise ValueError("The flat-field selection table contains no frame rows.")
    output_numbers: list[int] = [int(output) for output, _, _ in rows]
    if output_numbers != list(range(len(rows))):
        raise ValueError("Flat-field output TIFF numbers must be sequential from zero.")
    times_seconds: list[int] = [int(time) for _, time, _ in rows]
    frame_indices: list[int] = [int(index) for _, _, index in rows]
    if [round(time * expected_fps) for time in times_seconds] != frame_indices:
        raise ValueError("Flat-field times and frame indices do not agree with the recorded FPS.")
    return expected_fps, times_seconds, frame_indices


def populate_flat_field_frames(
    raw_chunks: Path,
    output_dir: Path,
    readme_path: Path,
    *,
    overwrite: bool = False,
) -> None:
    """Extract the README-selected planetarium orientations from raw chunks.

    Parameters
    ----------
    raw_chunks
        Raw planetarium world-camera recording directory.
    output_dir
        Directory that receives sequential uncompressed TIFF files.
    readme_path
        README containing the reconstruction FPS and frame selections.
    overwrite
        Whether existing output TIFFs may be replaced.

    Returns
    -------
    None
    """
    # Resolve the source once so downstream video helpers receive an absolute path.
    raw_chunks = raw_chunks.resolve()
    # Load the frame rate, times, and authoritative indices from the README.
    expected_fps, times_seconds, expected_frame_indices = (
        read_flat_field_selection_from_readme(readme_path)
    )
    # Import video support only for the stages that reconstruct recordings.
    import video_io

    # The AVI is an intermediate only. Keeping it in a temporary directory avoids
    # creating a second large calibration artifact in either Dropbox or data/.
    with tempfile.TemporaryDirectory(prefix="populate_flat_field_") as temporary_dir:
        temporary_video: Path = Path(temporary_dir) / "planetarium_fielding_function.avi"
        # Reconstruct the contiguous recording and apply its per-frame digital gain,
        # matching the preprocessing used for the checked-in flat-field TIFFs.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            output_path=str(temporary_video),
            verbose=True,
            convert_to_seconds=True,
            fill_missing_frames=True,
            apply_digital_gain=True,
        )
        # The documented times map to fixed indices only at the original 180 fps.
        frames_per_second: float = float(
            video_io.inspect_video_FPS(str(temporary_video))
        )
        if not np.isclose(frames_per_second, expected_fps):
            raise ValueError(
                f"Expected a {expected_fps:g} fps flat-field video; "
                f"found {frames_per_second:g} fps. Review the documented frame selection."
            )
        # Convert the human-readable time selection to video indices and cross-check
        # it against the authoritative list in defineFlatFieldingFunction.m.
        frame_indices: list[int] = [
            round(time * frames_per_second) for time in times_seconds
        ]
        if frame_indices != expected_frame_indices:
            raise ValueError("The derived flat-field indices no longer match defineFlatFieldingFunction.m.")

        # Extract in selection order and store the results directly as 0.tiff-35.tiff.
        frames: np.ndarray = video_io.extract_frames_from_video(
            str(temporary_video), frame_indices, verbose=True, is_grayscale=True
        )
        write_tiff_stack(frames, output_dir, overwrite=overwrite)

## Configuration and execution

In [ ]:
# Choose whether the existing 36-frame TIFF stack may be replaced.
OVERWRITE_FLAT_FIELD_FRAMES: bool = False
# Point to the raw planetarium recording and its generated TIFF destination.
FLAT_FIELD_RAW_CHUNKS: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "fielding_function/planetarium_fielding_function_raw"
)
FLAT_FIELD_OUTPUT: Path = DATA_ROOT / "flatFieldingFunction" / "rawFrames"
# Read the reconstruction settings and frame selections from this README.
FLAT_FIELD_README: Path = DATA_ROOT / "flatFieldingFunction" / "README.md"

# Run the extraction with all stage inputs visible at the call site.
populate_flat_field_frames(
    FLAT_FIELD_RAW_CHUNKS,
    FLAT_FIELD_OUTPUT,
    FLAT_FIELD_README,
    overwrite=OVERWRITE_FLAT_FIELD_FRAMES,
)

# `data/radiometricCorrectionRGB/`

**What this is:** A paired calibration between the spectral radiance of a cloudy sky measured with a PR670 and raw IMX219 images of that same sky. It is used to derive multiplicative RGB radiometric weights.

**Where it comes from:** The world-camera chunks are in Dropbox under `FLIC_data/LightLoggerRadCal/W1P1M1/radiometricCorrectionRGB/cloudyDayRecording`. The notebook logic comes from `code/preprocessRecordingData/another_scratch.ipynb`. The PR670 file `CloudySkySPD_37degSolarElevation.mat` and the illustrative `cropExample.tiff` are separately archived measurement inputs; this notebook does not recreate them.

**What it does:** Builds a temporary video without digital-gain, response-linearization, or color-weight corrections, then extracts grayscale frames 8000 through 8009. Preserving the uncorrected sensor values is essential because the downstream calibration is estimating those corrections.

**Output:** `data/radiometricCorrectionRGB/rawFrames/0.tiff` through `9.tiff`. `defineRadiometricWeights.m` combines these frames with the PR670 SPD and writes `derived/radiometricCorrectionRGB.mat`.

## Function definitions

In [ ]:
def populate_radiometric_frames(
    raw_chunks: Path,
    output_dir: Path,
    frame_indices: list[int],
    *,
    overwrite: bool = False,
) -> None:
    """Extract ten uncorrected cloudy-sky frames for RGB radiometric fitting."""
    # Resolve the raw recording path before passing it to video reconstruction.
    raw_chunks = raw_chunks.resolve()
    # Import video support only when this reconstruction stage runs.
    import video_io

    # Use a disposable AVI so the reconstructed recording never becomes a data asset.
    with tempfile.TemporaryDirectory(prefix="populate_radiometric_") as temporary_dir:
        temporary_video: Path = Path(temporary_dir) / "cloudy_day_recording.avi"
        # Disable every camera correction because these frames are inputs used to
        # estimate those corrections downstream.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            str(temporary_video),
            apply_digital_gain=False,
            convert_to_seconds=True,
            fill_missing_frames=True,
            verbose=True,
            linearize_camera_responsivity=False,
            apply_color_weights=False,
        )
        # Preserve the Bayer mosaic as grayscale and rename frames by selection order.
        frames: np.ndarray = video_io.extract_frames_from_video(
            str(temporary_video), frame_indices, is_grayscale=True
        )
        # Write sequential uncompressed TIFFs while honoring the overwrite policy.
        write_tiff_stack(frames, output_dir, overwrite=overwrite)

## Configuration and execution

In [ ]:
# Choose whether existing cloudy-sky TIFFs may be replaced.
OVERWRITE_RADIOMETRIC_FRAMES: bool = False
# Point to the raw cloudy-sky recording and generated TIFF destination.
RADIOMETRIC_RAW_CHUNKS: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_data/LightLoggerRadCal/W1P1M1/"
    "radiometricCorrectionRGB/cloudyDayRecording"
)
RADIOMETRIC_OUTPUT: Path = DATA_ROOT / "radiometricCorrectionRGB" / "rawFrames"
# Select the ten consecutive raw video frames used by the RGB fit.
RADIOMETRIC_FRAME_INDICES: list[int] = list(range(8000, 8010))

# Run the extraction with all stage inputs visible at the call site.
populate_radiometric_frames(
    RADIOMETRIC_RAW_CHUNKS,
    RADIOMETRIC_OUTPUT,
    RADIOMETRIC_FRAME_INDICES,
    overwrite=OVERWRITE_RADIOMETRIC_FRAMES,
)

# `data/darkSignal/`

**What this is:** Raw Bayer dark frames acquired with the lens cap installed, the camera wrapped in a black shroud, and the room dark. Separate recordings cover five fixed AGC states because exposure and analog gain can change the camera's dark behavior.

**Where it comes from:** The canonical recordings are stored in Dropbox at `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/darkNoiseCalibrations`, which is the configured source below. Each child directory must be one raw world-camera chunk recording and should retain its metadata-rich `AGCstate*_AGain-*_DGain-*_E-*` name.

**What it does:** Loads each recording's world metadata, removes synthetic gap rows whose camera settings are NaN, and chooses ten physical-frame indices evenly across the remaining rows. Each selected timestamp is resolved with `chunk_io.find_nearest_neighbor` for the world camera, and the returned raw Bayer frame is written to TIFF. The resulting fixed selections are documented in this measurement's README. No video conversion, gain, response linearization, or color weighting is applied.

**Output:** One ten-frame directory per state beneath `data/darkSignal/`. The README is preserved. `defineDarkSignal.m` subsequently computes the median dark signal and writes `derived/darkSignal.mat`.

## Function definitions

In [ ]:
def populate_dark_signal_frames(
    raw_root: Path,
    output_dir: Path,
    frame_count: int,
    state_pattern: re.Pattern[str],
    expected_state_count: int,
    *,
    overwrite: bool = False,
) -> None:
    """Extract ten uncorrected dark frames from each of five fixed AGC states."""
    # Resolve the parent directory before enumerating its AGC-state recordings.
    raw_root = raw_root.resolve()

    # Select only directories whose names encode one of the expected AGC states.
    state_recordings: list[Path] = sorted(
        path
        for path in raw_root.iterdir()
        if path.is_dir() and state_pattern.match(path.name)
    )
    # Require the configured number of fixed-state recordings.
    if len(state_recordings) != expected_state_count:
        raise ValueError(
            f"Expected {expected_state_count} AGC-state recording directories under "
            f"{raw_root}; found {len(state_recordings)}."
        )

    # Process each state independently. Synthetic gap rows have timestamps but
    # NaN camera settings, so exclude them before selecting physical frames.
    for recording_path in state_recordings:
        # Load world timestamps and camera settings for this fixed AGC state.
        world_metadata = world_util.world_metadata_from_chunks(
            str(recording_path), convert_to_seconds=True, verbose=False
        )
        # Treat every non-timestamp column as part of the captured camera state.
        setting_columns: list[str] = [
            column for column in world_metadata.columns if column != "timestamp"
        ]
        # Remove synthetic gap rows whose camera settings are entirely missing.
        captured_metadata = world_metadata.loc[
            ~world_metadata[setting_columns].isna().all(axis=1)
        ].reset_index(drop=True)

        # Spanning 0 through count-1 samples the full acquisition uniformly.
        recording_frame_count: int = len(captured_metadata)
        frame_indices: list[int] = [
            int(index)
            for index in np.linspace(
                0, recording_frame_count - 1, frame_count, dtype=np.int64
            )
        ]
        # Resolve each selected physical frame's timestamp through the shared
        # nearest-neighbor reader. Dark recordings need only the world frame.
        frames: np.ndarray = np.stack([
            chunk_io.find_nearest_neighbor(
                str(recording_path),
                float(captured_metadata.iloc[frame_index]["timestamp"]),
                sensors="W",
            )["W"]["value"]
            for frame_index in frame_indices
        ])
        # Keep each AGC state in its own metadata-rich output directory.
        write_tiff_stack(
            frames, output_dir / recording_path.name, overwrite=overwrite
        )

## Configuration and execution

In [5]:
# Choose whether existing dark-frame TIFFs may be replaced.
OVERWRITE_DARK_SIGNAL_FRAMES: bool = True
# Point to the parent of the five fixed-AGC recordings and their output root.
DARK_SIGNAL_RAW_ROOT: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "darkSignal"
)
DARK_SIGNAL_OUTPUT: Path = DATA_ROOT / "darkSignal"
# Select ten evenly spaced frames from each of five expected AGC states.
DARK_SIGNAL_FRAME_COUNT: int = 10
DARK_SIGNAL_EXPECTED_STATE_COUNT: int = 5
DARK_SIGNAL_STATE_PATTERN: re.Pattern[str] = re.compile(r"^AGCstate[1-5](?:_|$)")

# Run the extraction with all stage inputs visible at the call site.
populate_dark_signal_frames(
    DARK_SIGNAL_RAW_ROOT,
    DARK_SIGNAL_OUTPUT,
    DARK_SIGNAL_FRAME_COUNT,
    DARK_SIGNAL_STATE_PATTERN,
    DARK_SIGNAL_EXPECTED_STATE_COUNT,
    overwrite=OVERWRITE_DARK_SIGNAL_FRAMES,
)

# Root-level files in `data/`

The top level of `data/` contains several MAT files rather than another directory. Only the AGC-to-illuminance file came from one of the consolidated Python notebooks.

## `empircalAGCAndIlluminance.mat`

**Source:** Raw `GKA` recordings from the 2026 scripted indoor/outdoor dataset mounted at `/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026`.

**Operation:** Run `defineMSIlluminanceToAGCLag.py` to write the shared lag under `derived/`, then process the raw recordings in memory with that fixed lag, discard each recording's initial transient samples, retain finite positive samples below the configured saturation limit, and write the matched linear-scale camera-score/illuminance point cloud.

**Output:** `deriveEmpircalAGCAndIlluminance.py` reads `derived/MSIlluminanceToAGCLag.mat` and writes `data/empircalAGCAndIlluminance.mat` directly. The file contains the MATLAB struct `empiralAGC` with the fields `cameraScoreLinear`, `msIlluminance`, and `sharedLagSeconds`. The later MATLAB piecewise log-log fit is model fitting, not data population, so it is not run here.

## Function definitions

In [ ]:
def natural_sort_key(path_or_name: Path | str) -> list[int | str]:
    """Split names into text and integer pieces so FLIC_2 sorts before FLIC_10."""
    # Numeric tokens become integers while text tokens compare case-insensitively.
    return [int(piece) if piece.isdigit() else piece.lower() for piece in re.split(r"(\d+)", str(path_or_name))]


def populate_agc_to_illuminance(
    raw_root: Path,
    project_root: Path,
    lag_output: Path,
    data_output: Path,
    maximum_subjects: int,
    subjects_to_skip: set[str],
    maximum_saturation_percent: float,
    initial_samples_to_exclude: int,
    *,
    overwrite: bool = False,
) -> None:
    """Rebuild the empirical AGC/illuminance MAT file from raw subject recordings."""
    # Decide independently whether each derived MAT output requires generation.
    generate_lag: bool = overwrite or not lag_output.exists()
    generate_data: bool = overwrite or not data_output.exists()
    preserved_output_count: int = (
        0
        if overwrite
        else int(lag_output.exists()) + int(data_output.exists())
    )
    if not generate_lag and not generate_data:
        print(
            f"Generated 0 AGC MAT files; preserved {preserved_output_count} "
            "existing outputs."
        )
        return

    # Resolve the dataset root before enumerating subjects and activities.
    raw_root = raw_root.resolve()

    # Collect every activity/GKA recording for the first configured valid subjects.
    recording_paths: list[str] = []
    valid_subject_count: int = 0
    for subject_dir in sorted(raw_root.iterdir(), key=natural_sort_key):
        if valid_subject_count >= maximum_subjects:
            break
        # Ignore hidden/non-subject entries and any explicitly excluded subjects.
        if (
            not subject_dir.is_dir()
            or subject_dir.name.startswith(".")
            or subject_dir.name in subjects_to_skip
        ):
            continue
        # Each activity contributes its GKA directory as one analysis recording.
        for activity_dir in sorted(subject_dir.iterdir(), key=natural_sort_key):
            if not activity_dir.is_dir() or activity_dir.name.startswith("."):
                continue
            recording_path: Path = activity_dir / "GKA"
            if not recording_path.is_dir():
                raise FileNotFoundError(recording_path)
            recording_paths.append(str(recording_path))
        valid_subject_count += 1

    if not recording_paths:
        raise ValueError(f"No AGC recording directories were found under {raw_root}.")

    # Import the derivation modules after collecting the selected recordings.
    derive_module_dir: Path = project_root / "code" / "defineWorldCameraCalibration"
    data_prep_module_dir: Path = derive_module_dir / "dataPrep"
    for import_path in (derive_module_dir, data_prep_module_dir):
        if str(import_path) not in sys.path:
            sys.path.insert(0, str(import_path))
    import defineMSIlluminanceToAGCLag
    import deriveEmpircalAGCAndIlluminance
    importlib.reload(defineMSIlluminanceToAGCLag)
    importlib.reload(deriveEmpircalAGCAndIlluminance)

    # Generate each MAT output only when it is missing or replacement is enabled.
    lag_path: Path = lag_output
    generated_output_count: int = 0
    if generate_lag:
        lag_result: Any = defineMSIlluminanceToAGCLag.derive_agc_lag(
            recording_paths, output_path=lag_output
        )
        lag_path = Path(lag_result.output_path)
        generated_output_count += 1
    if generate_data:
        deriveEmpircalAGCAndIlluminance.derive_empircal_agc_and_illuminance(
            recording_paths,
            lag_path=lag_path,
            output_path=data_output,
            maximum_saturation_percent=maximum_saturation_percent,
            initial_samples_to_exclude=initial_samples_to_exclude,
        )
        generated_output_count += 1
    print(
        f"Generated {generated_output_count} AGC MAT files; preserved "
        f"{preserved_output_count} existing outputs."
    )

## Configuration and execution

In [ ]:
# Choose whether the lag and empirical AGC MAT files may be replaced.
OVERWRITE_AGC_TO_ILLUMINANCE: bool = False
# Point to the scripted dataset that supplies subject/activity recordings.
AGC_RAW_ROOT: Path = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
# Limit the analysis cohort and exclude known unsuitable subjects.
AGC_MAXIMUM_SUBJECTS: int = 4
AGC_SUBJECTS_TO_SKIP: set[str] = {"FLIC_18"}
# Configure sample filtering before the empirical relationship is written.
AGC_MAXIMUM_SATURATION_PERCENT: float = 40.0
AGC_INITIAL_SAMPLES_TO_EXCLUDE: int = 100
# Name the two MAT outputs produced by this stage.
AGC_DATA_OUTPUT: Path = DATA_ROOT / "empircalAGCAndIlluminance.mat"
AGC_LAG_OUTPUT: Path = PROJECT_ROOT / "derived" / "MSIlluminanceToAGCLag.mat"

# Run the derivation with all stage inputs visible at the call site.
populate_agc_to_illuminance(
    AGC_RAW_ROOT,
    PROJECT_ROOT,
    AGC_LAG_OUTPUT,
    AGC_DATA_OUTPUT,
    AGC_MAXIMUM_SUBJECTS,
    AGC_SUBJECTS_TO_SKIP,
    AGC_MAXIMUM_SATURATION_PERCENT,
    AGC_INITIAL_SAMPLES_TO_EXCLUDE,
    overwrite=OVERWRITE_AGC_TO_ILLUMINANCE,
)

# Validate the populated data tree

This read-only audit checks the expected counts and key files after any population sections have run. It deliberately ignores deprecated artifacts, hidden Finder metadata, and Python cache files.

In [ ]:
def numbered_tiff_count(directory: Path) -> int:
    """Count sequential data TIFFs while ignoring README and hidden support files."""
    # Only numeric stems belong to the zero-based frame stacks generated above.
    return len([path for path in directory.glob("*.tiff") if path.stem.isdigit()])


checks: dict[str, bool] = {
    "curated example MAT files": {
        path.name for path in EXAMPLE_OUTPUT_DIR.glob("*_AGCandMS_*.mat")
    } == {
        "indoor_AGCandMS_01.mat",
        "indoor_AGCandMS_02.mat",
        "indoor_AGCandMS_03.mat",
        "outdoor_AGCandMS_01.mat",
        "outdoor_AGCandMS_02.mat",
        "outdoor_AGCandMS_03.mat",
        "planetarium_AGCandMS_01.mat",
    },
    "numbered indoor/outdoor Macbeth ColorChecker MAT files": all(
        (
            MACBETH_OUTPUT_DIR / condition / str(measurement)
            / "lightLogger" / "close_AGCandMS_01.mat"
        ).is_file()
        for condition, measurement in read_macbeth_frame_selections(
            MACBETH_OUTPUT_DIR / "README.md", MACBETH_CONDITIONS
        )
    ),
    "AGC-by-NDF README": (AGC_NDF_OUTPUT_ROOT / "README.md").is_file(),
    "AGC-by-NDF PR670 directory": (AGC_NDF_OUTPUT_ROOT / "PR670").is_dir(),
    "three AGC-by-NDF MAT files per level": all(
        {path.name for path in (AGC_NDF_OUTPUT_ROOT / "worldCamera" / f"NDF{ndf_level}").glob("*_AGCandMS_*.mat")}
        == {
            f"NDF{ndf_level}_AGCandMS_{selection_number:02d}.mat"
            for selection_number in range(1, AGC_NDF_SELECTIONS_PER_LEVEL + 1)
        }
        for ndf_level in AGC_NDF_LEVELS
    ),
    "example-image README": (EXAMPLE_OUTPUT_DIR / "README.md").is_file(),
    "example-image DGain notebook": (EXAMPLE_OUTPUT_DIR / "find_DGain.ipynb").is_file(),
    "fisheye images": numbered_tiff_count(FISHEYE_IMAGE_OUTPUT) == 47,
    "README-selected flat-field frames": numbered_tiff_count(
        FLAT_FIELD_OUTPUT
    ) == len(read_flat_field_selection_from_readme(FLAT_FIELD_README)[1]),
    "radiometric frames": numbered_tiff_count(RADIOMETRIC_OUTPUT) == 10,
    "radiometric README": (DATA_ROOT / "radiometricCorrectionRGB" / "README.md").is_file(),
    "five dark-signal states": len([path for path in DARK_SIGNAL_OUTPUT.iterdir() if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)]) == DARK_SIGNAL_EXPECTED_STATE_COUNT,
    "ten frames in every dark-signal state": all(
        numbered_tiff_count(path) == DARK_SIGNAL_FRAME_COUNT
        for path in DARK_SIGNAL_OUTPUT.iterdir()
        if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)
    ),
    "AGC-to-illuminance MAT": AGC_DATA_OUTPUT.is_file(),
    "AS7341 sensitivity MAT": (DATA_ROOT / "ASM7341_spectralSensitivity.mat").is_file(),
    "IMX219 sensitivity MAT": (DATA_ROOT / "IMX219_spectralSensitivity.mat").is_file(),
    "camera-linearity MAT": (DATA_ROOT / "camera_linearity_ND0_ND0p4_rgb_means.mat").is_file(),
}

for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {description}")

if not all(checks.values()):
    raise AssertionError("One or more expected calibration-data checks failed.")